# Week 1 Baseline v2 — RED-CNN on Mayo + Cross-Vendor Test on Piglet

**v2 changes:** Real paired Quarter-Dose ↔ Full-Dose PNG loading (no noise simulation). Patient-level splits matching CAFM paper for direct comparability.

**Goal:** Reproduce RED-CNN (Chen 2017, IEEE TMI) on Mayo, then evaluate on Piglet (GE scanner). The Mayo→Piglet PSNR gap is the motivation experiment for the TTA method developed in subsequent weeks.

**Success criteria:**
- Mayo validation PSNR ≥ 35 dB at 25% dose (Quarter Dose)
- Measurable Mayo→Piglet PSNR drop (expect 2-5 dB) — this is the paper's hook

**Datasets:**
- Mayo (Siemens):  `/media/ant-pc/HDD/datasets/ct-low-dose-reconstruction/Preprocessed_512x512/512/{Full Dose,Quarter Dose}/1mm/Soft Kernel (B30)/L{ID}/*.png`
- Piglet (GE):     `/media/ant-pc/HDD/datasets/piglet/piglet/DICOM/PA0/ST0/SE{N}/`

**Patient splits (matching CAFM):**
- Train (6): L067, L096, L109, L286, L310, L333
- Val (2):   L192, L506
- Test held-out (2): L143, L291 — *not used in Week 1*


## 0. Setup


In [ ]:
import os, glob, json, time, re
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import pydicom
from skimage.metrics import structural_similarity as ssim_skimage

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU:    {torch.cuda.get_device_name(0)}')
    print(f'VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

torch.manual_seed(42)
np.random.seed(42)


In [ ]:
CONFIG = {
    # Paths
    'mayo_root':   '/media/ant-pc/HDD/datasets/ct-low-dose-reconstruction/Preprocessed_512x512/512',
    'piglet_root': '/media/ant-pc/HDD/datasets/piglet/piglet/DICOM/PA0/ST0',
    'output_dir':  '/media/ant-pc/HDD/experiments/week1_baseline',

    # Mayo structure (kernel B30 = soft, standard for Mayo LDCT)
    'mayo_thickness':  '1mm',
    'mayo_kernel':     'Soft Kernel (B30)',
    'train_patients':  ['L067', 'L096', 'L109', 'L286', 'L310', 'L333'],
    'val_patients':    ['L192', 'L506'],

    # Training
    'batch_size':         16,
    'patch_size':         64,
    'patches_per_image':  10,
    'lr':                 1e-4,
    'epochs':             50,
    'val_every':          5,
    'num_workers':        4,

    # HU normalization (used only for Piglet DICOM)
    'hu_min': -1024.0,
    'hu_max':  3071.0,
}

os.makedirs(CONFIG['output_dir'], exist_ok=True)
print(json.dumps(CONFIG, indent=2))


## 1. Data Inspection

Verify Mayo PNG file format (bit depth, range) and Piglet DICOM series metadata.


In [ ]:
def inspect_mayo_png(root, train_patients, kernel, thickness):
    """Probe one PNG from Quarter Dose and one from Full Dose to verify format."""
    root = Path(root)
    patient = train_patients[0]
    qd_dir = root / 'Quarter Dose' / thickness / kernel / patient
    fd_dir = root / 'Full Dose'    / thickness / kernel / patient

    print(f'Probing patient {patient}')
    for tag, d in [('Quarter Dose', qd_dir), ('Full Dose', fd_dir)]:
        if not d.exists():
            print(f'  ❌ Missing: {d}')
            continue
        files = sorted(d.glob('*.png'))
        print(f'  {tag}: {len(files)} PNG files at {d}')
        if files:
            img = np.array(Image.open(files[0]))
            print(f'    First file: {files[0].name}')
            print(f'    dtype={img.dtype}  shape={img.shape}  range=[{img.min()}, {img.max()}]')

    # Count totals across patients
    total_qd = total_fd = 0
    for pat in train_patients:
        qd = root / 'Quarter Dose' / thickness / kernel / pat
        fd = root / 'Full Dose'    / thickness / kernel / pat
        if qd.exists(): total_qd += len(list(qd.glob('*.png')))
        if fd.exists(): total_fd += len(list(fd.glob('*.png')))
    print(f'\nAcross {len(train_patients)} train patients: {total_qd} QD, {total_fd} FD')

inspect_mayo_png(CONFIG['mayo_root'], CONFIG['train_patients'],
                 CONFIG['mayo_kernel'], CONFIG['mayo_thickness'])


In [ ]:
def inspect_piglet(root):
    """Build Piglet series metadata table."""
    root = Path(root)
    if not root.exists():
        print(f'❌ Path does not exist: {root}')
        return None

    series_dirs = sorted(
        [d for d in root.iterdir() if d.is_dir() and d.name.startswith('SE')],
        key=lambda x: int(x.name[2:])
    )

    info = []
    for sd in series_dirs:
        files = sorted(sd.iterdir())
        if not files:
            continue
        try:
            ds = pydicom.dcmread(str(files[0]), stop_before_pixels=True)
            info.append({
                'series':           sd.name,
                'n_slices':         len(files),
                'mAs':              float(ds.get('Exposure', 0)),
                'kVp':              float(ds.get('KVP', 0)),
                'kernel':           str(ds.get('ConvolutionKernel', '?')),
                'slice_thickness':  float(ds.get('SliceThickness', 0)),
            })
        except Exception as e:
            print(f'  Skip {sd.name}: {e}')

    return info

piglet_info = inspect_piglet(CONFIG['piglet_root'])
print(f'Piglet series: {len(piglet_info)}')
print()
print(f"{'Series':<8} {'Slices':<8} {'mAs':<6} {'kVp':<6} {'Kernel':<14} {'Thick':<6}")
print('-' * 55)
for s in piglet_info:
    print(f"{s['series']:<8} {s['n_slices']:<8} {s['mAs']:<6.0f} {s['kVp']:<6.0f} {s['kernel']:<14} {s['slice_thickness']:<6.1f}")


In [ ]:
# Pick Piglet pairs: highest-mAs series as reference, mid-low mAs as test input
piglet_with_mas = [s for s in piglet_info if s['mAs'] > 0]
piglet_sorted   = sorted(piglet_with_mas, key=lambda x: -x['mAs'])

PIGLET_REF = piglet_sorted[0]['series']
low_candidates = [s for s in piglet_sorted if s['mAs'] < piglet_sorted[0]['mAs']]
PIGLET_LOW = (low_candidates[len(low_candidates) // 2]['series']
              if low_candidates else None)

print(f'Reference (high-mAs): {PIGLET_REF}  ({piglet_sorted[0]["mAs"]:.0f} mAs)')
print(f'Low-dose (test input): {PIGLET_LOW}')

with open(os.path.join(CONFIG['output_dir'], 'piglet_series.json'), 'w') as f:
    json.dump({'ref': PIGLET_REF, 'low': PIGLET_LOW, 'all': piglet_info}, f, indent=2)


## 2. Datasets

### MayoPNGDataset
Loads **real paired** Quarter-Dose ↔ Full-Dose PNG slices. Matches files across QD/FD folders by extracting the slice number from filename (`L109_QD_1_1.CT.0003.**0190**.<datetime>.png`).

### PigletDataset
Loads DICOM slices from two series (high-mAs reference, low-mAs test input), matches by `InstanceNumber`, converts to HU.


In [ ]:
def extract_slice_num(path):
    """
    Filename like 'L109_QD_1_1.CT.0003.0190.2015.12.23.17.53.56.544757.125867905.png'
    The token after 'CT.0003.' is the slice number (e.g. '0190').
    """
    stem  = path.stem
    parts = stem.split('.')
    try:
        ct_idx = parts.index('CT')
        # parts[ct_idx + 1] is something like '0003'
        # parts[ct_idx + 2] is the slice number, e.g. '0190'
        return int(parts[ct_idx + 2])
    except (ValueError, IndexError):
        return None


def load_png_normalized(path):
    """Load PNG as float32 in [0, 1]. Handles 8-bit and 16-bit PNGs."""
    img = np.array(Image.open(path))
    if img.ndim == 3:
        img = img.mean(axis=2)  # RGB → grayscale (fallback)
    if img.dtype == np.uint8:
        return img.astype(np.float32) / 255.0
    if img.dtype == np.uint16:
        return img.astype(np.float32) / 65535.0
    img = img.astype(np.float32)
    # Generic: normalize to [0,1] based on observed range
    if img.max() > img.min():
        img = (img - img.min()) / (img.max() - img.min())
    return img


class MayoPNGDataset(Dataset):
    """Real paired Quarter-Dose ↔ Full-Dose Mayo dataset (PNG)."""

    def __init__(self, root, patients, kernel, thickness,
                 patch_size=64, patches_per_image=10, mode='train'):
        self.root              = Path(root)
        self.patch_size        = patch_size
        self.patches_per_image = patches_per_image
        self.mode              = mode

        self.pairs = []
        n_missing = 0
        for pat in patients:
            qd_dir = self.root / 'Quarter Dose' / thickness / kernel / pat
            fd_dir = self.root / 'Full Dose'    / thickness / kernel / pat
            if not (qd_dir.exists() and fd_dir.exists()):
                print(f'  Skip {pat}: missing QD or FD folder')
                continue
            qd_files = {extract_slice_num(f): f for f in qd_dir.glob('*.png')}
            fd_files = {extract_slice_num(f): f for f in fd_dir.glob('*.png')}
            qd_files = {k: v for k, v in qd_files.items() if k is not None}
            fd_files = {k: v for k, v in fd_files.items() if k is not None}
            common   = sorted(set(qd_files) & set(fd_files))
            for s in common:
                self.pairs.append((qd_files[s], fd_files[s], pat, s))
            n_missing += abs(len(qd_files) - len(fd_files))

        print(f'Mayo [{mode}]: {len(self.pairs)} paired slices from {len(patients)} patients',
              f'(unmatched: {n_missing})' if n_missing else '')

    def __len__(self):
        if self.mode == 'train':
            return len(self.pairs) * self.patches_per_image
        return len(self.pairs)

    def __getitem__(self, idx):
        pair_idx = idx % len(self.pairs)
        qd_path, fd_path, _patient, _slice = self.pairs[pair_idx]
        ldct = load_png_normalized(qd_path)
        ndct = load_png_normalized(fd_path)

        if self.mode == 'train':
            h, w = ndct.shape
            ph   = self.patch_size
            if h > ph and w > ph:
                top  = np.random.randint(0, h - ph)
                left = np.random.randint(0, w - ph)
                ndct = ndct[top:top+ph, left:left+ph]
                ldct = ldct[top:top+ph, left:left+ph]

        return (torch.from_numpy(ldct).unsqueeze(0).float(),
                torch.from_numpy(ndct).unsqueeze(0).float())


In [ ]:
def normalize_hu(img, hu_min, hu_max):
    img = (img.astype(np.float32) - hu_min) / (hu_max - hu_min)
    return np.clip(img, 0.0, 1.0)


class PigletDataset(Dataset):
    """Piglet cross-vendor test. Pairs slices across two series by InstanceNumber."""

    def __init__(self, root, ref_series, low_series,
                 hu_min=-1024, hu_max=3071, max_slices=300):
        self.root   = Path(root)
        self.hu_min = hu_min
        self.hu_max = hu_max

        ref_list = self._load_series_files(ref_series)
        low_list = self._load_series_files(low_series)
        ref_map  = dict(ref_list)
        low_map  = dict(low_list)
        common   = sorted(set(ref_map) & set(low_map))[:max_slices]
        self.ref_files = [ref_map[i] for i in common]
        self.low_files = [low_map[i] for i in common]
        print(f'Piglet: ref={ref_series}  low={low_series}  matched slices={len(common)}')

    def _load_series_files(self, series_name):
        out = []
        for f in (self.root / series_name).iterdir():
            try:
                ds = pydicom.dcmread(str(f), stop_before_pixels=True)
                out.append((int(ds.InstanceNumber), str(f)))
            except Exception:
                continue
        return sorted(out)

    def _load_dicom(self, path):
        ds        = pydicom.dcmread(path)
        img       = ds.pixel_array.astype(np.float32)
        slope     = float(ds.get('RescaleSlope', 1.0))
        intercept = float(ds.get('RescaleIntercept', 0.0))
        return normalize_hu(img * slope + intercept, self.hu_min, self.hu_max)

    def __len__(self):
        return len(self.ref_files)

    def __getitem__(self, idx):
        low = self._load_dicom(self.low_files[idx])
        ref = self._load_dicom(self.ref_files[idx])
        return (torch.from_numpy(low).unsqueeze(0).float(),
                torch.from_numpy(ref).unsqueeze(0).float())


In [ ]:
# Build datasets
train_set = MayoPNGDataset(
    CONFIG['mayo_root'], CONFIG['train_patients'],
    CONFIG['mayo_kernel'], CONFIG['mayo_thickness'],
    patch_size=CONFIG['patch_size'],
    patches_per_image=CONFIG['patches_per_image'],
    mode='train',
)
val_set = MayoPNGDataset(
    CONFIG['mayo_root'], CONFIG['val_patients'],
    CONFIG['mayo_kernel'], CONFIG['mayo_thickness'],
    patch_size=CONFIG['patch_size'],
    patches_per_image=1,
    mode='val',
)
piglet_test = PigletDataset(
    CONFIG['piglet_root'], PIGLET_REF, PIGLET_LOW,
    hu_min=CONFIG['hu_min'], hu_max=CONFIG['hu_max'], max_slices=300,
)

print(f'\nTrain samples (with patch sampling): {len(train_set):,}')
print(f'Val   samples: {len(val_set):,}')
print(f'Piglet test samples: {len(piglet_test):,}')


In [ ]:
# Sanity-check visualization
fig, axes = plt.subplots(2, 3, figsize=(13, 8))

# Mayo val sample (no patch cropping when mode='val')
ldct, ndct = val_set[0]
axes[0, 0].imshow(ldct[0].numpy(), cmap='gray', vmin=0, vmax=1)
axes[0, 0].set_title('Mayo Quarter Dose (real LDCT)')
axes[0, 1].imshow(ndct[0].numpy(), cmap='gray', vmin=0, vmax=1)
axes[0, 1].set_title('Mayo Full Dose (NDCT)')
axes[0, 2].imshow((ndct - ldct)[0].numpy(), cmap='RdBu', vmin=-0.2, vmax=0.2)
axes[0, 2].set_title('NDCT − LDCT')

low, ref = piglet_test[len(piglet_test) // 2]
axes[1, 0].imshow(low[0].numpy(), cmap='gray', vmin=0, vmax=1)
axes[1, 0].set_title(f'Piglet {PIGLET_LOW} (low mAs)')
axes[1, 1].imshow(ref[0].numpy(), cmap='gray', vmin=0, vmax=1)
axes[1, 1].set_title(f'Piglet {PIGLET_REF} (high mAs)')
axes[1, 2].imshow((ref - low)[0].numpy(), cmap='RdBu', vmin=-0.2, vmax=0.2)
axes[1, 2].set_title('Ref − Low')

for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'sample_check.png'), dpi=100)
plt.show()


## 3. RED-CNN Model

Chen et al. 2017, IEEE TMI 36(12):2524-2535. Residual encoder-decoder with 5 conv + 5 deconv layers, 96 filters, 5×5 kernels, symmetric shortcut connections. ~1.85M parameters.


In [ ]:
class REDCNN(nn.Module):
    """Chen et al. 2017, IEEE TMI 36(12):2524-2535."""

    def __init__(self, in_ch=1, out_ch=1, n_filters=96):
        super().__init__()
        kw = dict(kernel_size=5, stride=1, padding=0)

        self.conv1 = nn.Conv2d(in_ch,     n_filters, **kw)
        self.conv2 = nn.Conv2d(n_filters, n_filters, **kw)
        self.conv3 = nn.Conv2d(n_filters, n_filters, **kw)
        self.conv4 = nn.Conv2d(n_filters, n_filters, **kw)
        self.conv5 = nn.Conv2d(n_filters, n_filters, **kw)

        self.deconv1 = nn.ConvTranspose2d(n_filters, n_filters, **kw)
        self.deconv2 = nn.ConvTranspose2d(n_filters, n_filters, **kw)
        self.deconv3 = nn.ConvTranspose2d(n_filters, n_filters, **kw)
        self.deconv4 = nn.ConvTranspose2d(n_filters, n_filters, **kw)
        self.deconv5 = nn.ConvTranspose2d(n_filters, out_ch,    **kw)
        self.relu    = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x
        c1 = self.relu(self.conv1(x))
        c2 = self.relu(self.conv2(c1))
        c3 = self.relu(self.conv3(c2))
        c4 = self.relu(self.conv4(c3))
        c5 = self.relu(self.conv5(c4))
        d1 = self.relu(self.deconv1(c5) + c4)
        d2 = self.relu(self.deconv2(d1))
        d3 = self.relu(self.deconv3(d2) + c2)
        d4 = self.relu(self.deconv4(d3))
        d5 = self.deconv5(d4)
        return self.relu(d5 + residual)


model    = REDCNN().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'RED-CNN parameters: {n_params:,}  ({n_params/1e6:.2f}M)')

with torch.no_grad():
    x = torch.randn(2, 1, 64, 64).to(device)
    y = model(x)
print(f'Input: {tuple(x.shape)}  →  Output: {tuple(y.shape)}')
assert y.shape == x.shape, 'Shape mismatch'


## 4. Metrics + Training Loop


In [ ]:
def compute_psnr(pred, target, data_range=1.0):
    mse = F.mse_loss(pred, target).item()
    if mse <= 0:
        return float('inf')
    return 10.0 * np.log10((data_range ** 2) / mse)


def compute_ssim(pred, target, data_range=1.0):
    p = pred.detach().cpu().numpy().squeeze()
    t = target.detach().cpu().numpy().squeeze()
    if p.ndim == 3:
        return float(np.mean([ssim_skimage(p[i], t[i], data_range=data_range)
                              for i in range(p.shape[0])]))
    return float(ssim_skimage(p, t, data_range=data_range))


@torch.no_grad()
def evaluate(model, loader, device, max_batches=None):
    model.eval()
    psnrs, ssims = [], []
    for i, (ldct, ndct) in enumerate(tqdm(loader, desc='Eval', leave=False)):
        if max_batches is not None and i >= max_batches:
            break
        ldct = ldct.to(device); ndct = ndct.to(device)
        pred = model(ldct)
        for b in range(pred.shape[0]):
            psnrs.append(compute_psnr(pred[b:b+1], ndct[b:b+1]))
            ssims.append(compute_ssim(pred[b:b+1], ndct[b:b+1]))
    return float(np.mean(psnrs)), float(np.mean(ssims))


@torch.no_grad()
def evaluate_input(loader, device):
    """No-denoise baseline: just measure input vs target PSNR/SSIM."""
    psnrs, ssims = [], []
    for ldct, ndct in tqdm(loader, desc='Raw input', leave=False):
        ldct = ldct.to(device); ndct = ndct.to(device)
        psnrs.append(compute_psnr(ldct, ndct))
        ssims.append(compute_ssim(ldct, ndct))
    return float(np.mean(psnrs)), float(np.mean(ssims))


In [ ]:
def train_model(model, train_loader, val_loader, *, epochs, lr, device, save_path, val_every=5):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.L1Loss()
    history   = {'train_loss': [], 'val_psnr': [], 'val_ssim': []}
    best_psnr = -1.0

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0; n_batches = 0
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}', leave=False)
        for ldct, ndct in pbar:
            ldct = ldct.to(device, non_blocking=True)
            ndct = ndct.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            pred = model(ldct)
            loss = criterion(pred, ndct)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item(); n_batches += 1
            pbar.set_postfix(loss=f'{loss.item():.4f}')
        scheduler.step()

        avg_loss = epoch_loss / max(1, n_batches)
        history['train_loss'].append(avg_loss)

        if (epoch + 1) % val_every == 0 or epoch == epochs - 1:
            vp, vs = evaluate(model, val_loader, device, max_batches=50)
            history['val_psnr'].append([epoch + 1, vp])
            history['val_ssim'].append([epoch + 1, vs])
            print(f'Epoch {epoch+1:3d}  loss={avg_loss:.4f}  val PSNR={vp:.2f} dB  SSIM={vs:.4f}')
            if vp > best_psnr:
                best_psnr = vp
                torch.save({'model': model.state_dict(),
                            'epoch': epoch + 1, 'psnr': vp, 'ssim': vs}, save_path)

    print(f'\nBest val PSNR: {best_psnr:.2f} dB → {save_path}')
    return history


In [ ]:
train_loader = DataLoader(train_set, batch_size=CONFIG['batch_size'],
                          shuffle=True, num_workers=CONFIG['num_workers'],
                          pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_set,   batch_size=1, shuffle=False, num_workers=2)

model     = REDCNN().to(device)
save_path = os.path.join(CONFIG['output_dir'], 'redcnn_mayo_best.pt')

t0 = time.time()
history = train_model(
    model, train_loader, val_loader,
    epochs=CONFIG['epochs'], lr=CONFIG['lr'], device=device,
    save_path=save_path, val_every=CONFIG['val_every'],
)
print(f'Training time: {(time.time() - t0)/60:.1f} min')

with open(os.path.join(CONFIG['output_dir'], 'history.json'), 'w') as f:
    json.dump(history, f, indent=2)


## 5. In-Domain Evaluation (Mayo)


In [ ]:
ckpt = torch.load(save_path, map_location=device)
model.load_state_dict(ckpt['model'])
print(f"Loaded ckpt epoch={ckpt['epoch']}  PSNR={ckpt['psnr']:.2f} dB  SSIM={ckpt['ssim']:.4f}")

mayo_in_psnr,  mayo_in_ssim  = evaluate_input(val_loader, device)
mayo_out_psnr, mayo_out_ssim = evaluate(model, val_loader, device)

print()
print(f'Mayo LDCT input (no denoise): PSNR={mayo_in_psnr:.2f} dB  SSIM={mayo_in_ssim:.4f}')
print(f'Mayo RED-CNN   (denoised):    PSNR={mayo_out_psnr:.2f} dB  SSIM={mayo_out_ssim:.4f}')
print(f'Improvement:                  +{mayo_out_psnr - mayo_in_psnr:.2f} dB')

results = {'mayo': {
    'input_psnr':    mayo_in_psnr,  'input_ssim':    mayo_in_ssim,
    'denoised_psnr': mayo_out_psnr, 'denoised_ssim': mayo_out_ssim,
    'improvement_db': mayo_out_psnr - mayo_in_psnr,
}}


## 6. Cross-Vendor Evaluation (Piglet, GE) — KEY EXPERIMENT

RED-CNN trained on Mayo (Siemens) tested on Piglet (GE). Significant PSNR drop or degraded visuals = paper motivation confirmed.


In [ ]:
piglet_loader = DataLoader(piglet_test, batch_size=1, shuffle=False, num_workers=2)

pig_in_psnr,  pig_in_ssim  = evaluate_input(piglet_loader, device)
pig_out_psnr, pig_out_ssim = evaluate(model, piglet_loader, device)

print(f'Piglet raw (no denoise):       PSNR={pig_in_psnr:.2f} dB  SSIM={pig_in_ssim:.4f}')
print(f'Piglet RED-CNN (Mayo-trained): PSNR={pig_out_psnr:.2f} dB  SSIM={pig_out_ssim:.4f}')
print(f'Improvement:                   {pig_out_psnr - pig_in_psnr:+.2f} dB')

results['piglet'] = {
    'ref_series':    PIGLET_REF, 'low_series': PIGLET_LOW,
    'input_psnr':    pig_in_psnr,  'input_ssim':    pig_in_ssim,
    'denoised_psnr': pig_out_psnr, 'denoised_ssim': pig_out_ssim,
    'improvement_db': pig_out_psnr - pig_in_psnr,
}


## 7. Results Summary


In [ ]:
with open(os.path.join(CONFIG['output_dir'], 'results.json'), 'w') as f:
    json.dump(results, f, indent=2)

print('=' * 70)
print('WEEK 1 RESULTS')
print('=' * 70)
print(f"{'Setting':<35} {'PSNR (dB)':<12} {'SSIM':<10}")
print('-' * 70)
print(f"{'Mayo LDCT input (no denoise)':<35} {results['mayo']['input_psnr']:<12.2f} {results['mayo']['input_ssim']:<10.4f}")
print(f"{'Mayo RED-CNN (in-domain)':<35} {results['mayo']['denoised_psnr']:<12.2f} {results['mayo']['denoised_ssim']:<10.4f}")
print(f"{'Piglet raw (no denoise)':<35} {results['piglet']['input_psnr']:<12.2f} {results['piglet']['input_ssim']:<10.4f}")
print(f"{'Piglet RED-CNN (cross-vendor)':<35} {results['piglet']['denoised_psnr']:<12.2f} {results['piglet']['denoised_ssim']:<10.4f}")
print('-' * 70)

mayo_gain   = results['mayo']['improvement_db']
piglet_gain = results['piglet']['improvement_db']
gap_psnr    = results['mayo']['denoised_psnr'] - results['piglet']['denoised_psnr']
gap_drop    = mayo_gain - piglet_gain

print(f'Mayo denoising gain:    +{mayo_gain:.2f} dB')
print(f'Piglet denoising gain:  {piglet_gain:+.2f} dB')
print(f'Mayo→Piglet PSNR gap:   {gap_psnr:.2f} dB')
print(f'Gain degradation:       {gap_drop:.2f} dB  (paper-worthy if ≥ 2 dB)')
print('=' * 70)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], 'b-')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('L1 loss')
axes[0].set_title('Training loss'); axes[0].grid(alpha=0.3)

if history['val_psnr']:
    ve, vp = zip(*history['val_psnr'])
    axes[1].plot(ve, vp, 'go-')
    axes[1].axhline(35, color='r', linestyle='--', alpha=0.5, label='35 dB target')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('PSNR (dB)')
    axes[1].set_title('Mayo validation PSNR'); axes[1].grid(alpha=0.3); axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'training_curves.png'), dpi=120)
plt.show()


In [ ]:
# Qualitative: one Piglet slice
model.eval()
idx      = len(piglet_test) // 2
low, ref = piglet_test[idx]
with torch.no_grad():
    denoised = model(low.unsqueeze(0).to(device)).cpu().squeeze()

p_low = compute_psnr(low.unsqueeze(0).to(device), ref.unsqueeze(0).to(device))
p_den = compute_psnr(denoised.unsqueeze(0).unsqueeze(0).to(device),
                     ref.unsqueeze(0).to(device))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(low.squeeze(), cmap='gray', vmin=0, vmax=1)
axes[0].set_title(f'Piglet low-mAs ({PIGLET_LOW})\nPSNR vs ref: {p_low:.2f} dB')
axes[1].imshow(denoised,      cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'RED-CNN output (Mayo-trained)\nPSNR vs ref: {p_den:.2f} dB')
axes[2].imshow(ref.squeeze(), cmap='gray', vmin=0, vmax=1)
axes[2].set_title(f'Piglet high-mAs reference ({PIGLET_REF})')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'piglet_qualitative.png'), dpi=120)
plt.show()


## Week 1 Decision Points

| Check | Status | Action |
|---|---|---|
| Mayo val PSNR ≥ 35 dB | ✅ | Pipeline OK → Week 2 |
| Mayo val PSNR 33-35 dB | ⚠️ | Borderline — verify nothing's miscalibrated |
| Mayo val PSNR < 33 dB | ❌ | Pipeline bug — debug first |
| Piglet gain degradation ≥ 2 dB | ✅ | Cross-vendor story is paper-worthy |
| Piglet gain degradation 1-2 dB | ⚠️ | Weak but acceptable |
| Piglet gain degradation < 1 dB | ❌ | Need alternative cross-domain axis |

### What's deferred
- Sinogram-domain noise simulation (Week 2, using ASTRA 2.4.1)
- LoRA adapter for backbone freezing (Week 2)
- Test-time adaptation loop (Week 3)
- Other baselines: CTformer, CoreDiff (Week 4)
- Test patients L143, L291 (Week 4, for final reporting)

### Files produced in `output_dir`
- `redcnn_mayo_best.pt`
- `history.json`, `results.json`, `piglet_series.json`
- `sample_check.png`, `training_curves.png`, `piglet_qualitative.png`
